# T04 · MACELES — long-range electrostatics

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Neutrino155/MACE_tutorials/blob/main/MACE_extensions/T04-MACELES.ipynb)

**Lennard-Jones Centre · University of Cambridge · Atomic-Scale Modelling**

Follow a local MACE energy plus an explicit long-range electrostatic branch, then compare matched water-dimer interaction curves.

**Route:** architecture in [T00](T00-Extending-MACE.ipynb); training foundations in [Practice I](../MACE_in_practice_I/T01-MACE-Practice-I.ipynb) and [Practice II](../MACE_in_practice_II/T02-MACE-Practice-II.ipynb); optional theory in [MACE Theory](../MACE_advanced/T03-MACE-Theory.ipynb).

## Setup

Run the setup and import cells in order. Local Jupyter uses upstream ACEsuit/MACE `develop` (or `MACE_ROOT`); Colab fetches it and installs the LES dependency when needed. The setup cell uses CUDA when available and otherwise CPU; edit `device` there. See the [extension guide](README.md) for source revisions and data notes.

In [1]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
_original_showwarning = warnings.showwarning


def _quiet_user_and_deprecation_warnings(message, category, filename, lineno,
                                         file=None, line=None):
    if issubclass(category, (UserWarning, DeprecationWarning)):
        return
    _original_showwarning(message, category, filename, lineno, file=file, line=line)


warnings.showwarning = _quiet_user_and_deprecation_warnings

import subprocess
import sys
from pathlib import Path

tutorial_root = next(
    (root.resolve() for root in (Path.cwd(), *Path.cwd().parents)
     if (root / "MACE_extensions").is_dir()),
    None,
)
if tutorial_root is None:
    if not Path("/content").is_dir():
        raise FileNotFoundError("Open this notebook from the MACE_tutorials repository.")
    tutorial_root = Path("/content/MACE_tutorials")
    if not (tutorial_root / "MACE_extensions").is_dir():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", "main",
             "https://github.com/Neutrino155/MACE_tutorials.git", str(tutorial_root)],
            check=True,
        )

sys.path.insert(0, str(tutorial_root))
from MACE_extensions.scripts.notebook_setup import setup

setup_paths = setup(feature='les')
from MACE_extensions.scripts.bootstrap import ensure
ensure('les', source_root=Path(setup_paths["source_root"]))
asset_root = Path(setup_paths["asset_root"])
source_root = Path(setup_paths["source_root"])


upstream MACE source: /home/brad/repositories/mace-upstream-tutorial · Colab: False
MACE source ready: /home/brad/repositories/mace-upstream-tutorial


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from ase import Atoms
from ase.io import read
from IPython.display import display
from mace.calculators import MACECalculator
from mace.modules.extensions import MACELES

from MACE_extensions.scripts.notebook_tools import (
    read_source, show_source_hits, train_mace,
)
import importlib
from MACE_extensions.scripts import structure_viewer
importlib.reload(structure_viewer)
animate_structures = structure_viewer.animate_structures
show_structure = structure_viewer.show_structure
show_structure_gallery = structure_viewer.show_structure_gallery

device = "cuda" if torch.cuda.is_available() else "cpu"
device = "cpu"
inference_device = "cpu"
torch.set_num_threads(1)
dtype = "float32"
les_model_path = Path(setup_paths["les_model"])
local_model_path = Path(setup_paths["les_local_control"])

def require_model(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Model file not found: {path}")
    return path

print(f"Source: {source_root} · {device}")

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.showwarning = _quiet_user_and_deprecation_warnings


### Inspect example structures

Choose a teaching set from the menu, then rotate and zoom to inspect its geometry and available overlays.

In [ ]:
data_choices = {
    "BaTiO₃ soft-mode training data": "macefield_batio3_toy_train.extxyz",
    "Water dipole training data": "atomicdipoles_water_train.extxyz",
    "Fe₂ magnetic training data": "magnetic_fe2_train.extxyz",
    "Water dimer LES training data": "maceles_water_dimer_train.extxyz",
}

example_structures = {
    name: read(asset_root / "data" / filename, index=0)
    for name, filename in data_choices.items()
}
display(show_structure_gallery(
    example_structures,
    title="Inspect the teaching structures",
    default_choice='Water dimer LES training data',
))


## Extension 4 — MACELES: add a long-range energy branch

### 4.1 Architecture change

A finite-cutoff model has no direct message between fragments beyond its cutoff. MACELES keeps the local MACE energy and adds an electrostatic contribution evaluated from learned latent charges and optional multipoles. The LES solver combines these outputs with positions and the cell; forces are differentiated from the total energy. Latent charges are internal variables, not unique partial charges.

#### Long-range branch in the architecture

Readouts from final node features feed the LES solver in parallel with the local energy path. The new term joins the energy after message passing.

![MACELES architecture: local and long-range energy branches](./figures/architecture_maceles.png)

### 4.2 Model boundary

MACELES adds nonlocal electrostatic coupling, but it does not make every long-range chemical effect exact.

<details>
<summary><b>Background theory: assembling the LES energy</b></summary>

$$E_{\mathrm{total}}(R,\mathrm{cell})=E_{\mathrm{local}}(R)+E_{\mathrm{LES}}(q,\mu,Q,\alpha,\ldots;R,\mathrm{cell}).$$

Here $q$, $\mu$, $Q$ and $\alpha$ denote the latent quantities enabled in the configuration. The model aggregates readouts, applies configured scales and constraints, calls the LES library, and adds $E_{\mathrm{LES}}$ before calculating forces. Total-charge constraints, boundary conditions, solver settings and training coverage remain important.

</details>

<details>
<summary><b>Implementation notes: charge, boundaries and solver</b></summary>

- Keep the total charge and boundary conditions consistent with the LES configuration; inspect neutralizing-background or charge-constraint choices for periodic charged systems.
- A latent charge is not automatically an observable partial charge. Do not validate it by comparison with one arbitrary population analysis unless that is separately justified.
- For a periodic crystal, use a valid cell and the periodic LES path. For a molecule, state the non-periodic boundary convention.
- If optional multipoles or induced responses are enabled, check their equivariance, units, stability, and solver conditioning.
- MACELES requires the external <code>les</code> package. Upstream MACE keeps the pinned LES dependency in <code>requirements/les.txt</code>; it is not part of base MACE. The setup cell installs it when needed.
</details>

### 4.3 Follow the implementation

Trace the final-layer readouts, the `self.les(...)` call and where `les_energy` is added before force calculation.

In [ ]:
show_source_hits(
    MACELES.__init__,
    needles=("keep_last_layer_irreps", "from les import Les", "les_readouts", "use_dipoles", "use_quads"),
    context=5,
    max_chars=10000,
)
show_source_hits(
    MACELES.forward,
    needles=("node_qs", "les_q =", "self.les(", "total_energy += les_energy", "get_outputs("),
    context=6,
    max_chars=12000,
)

#### Source-reading task

Find which irreps support optional outputs, where quadrupoles are constrained, how nonperiodic cells are handled, and which energy terms the calculator exposes.

### 4.4 Make and inspect a water dimer

The water-dimer probe varies the O···O separation while holding the monomer geometries fixed. A compatible LES checkpoint can be used to view its latent charges and energy decomposition. The plot is a model diagnostic, not a benchmark unless compared to reference data and a matched local-only model.

In [ ]:
def water_monomer_at(origin=(0.0, 0.0, 0.0)):
    origin = np.asarray(origin, dtype=float)
    local = np.array([[0.0, 0.0, 0.0], [0.96, 0.0, 0.0], [-0.24, 0.93, 0.0]])
    return origin + local


def water_dimer(separation=3.0, box=20.0):
    first = water_monomer_at((7.0, 8.0, 8.0))
    second = water_monomer_at((7.0, 8.0, 8.0 + separation))
    atoms = Atoms("OH2OH2", positions=np.vstack([first, second]),
                  cell=[box, box, box], pbc=False)
    return atoms

dimer = water_dimer(3.0)
display(show_structure(dimer, title="Water dimer probe at 3.0 Å O···O separation"))

#### Inspect raw model outputs

This learning cell uses a private calculator method to expose latent charges and the LES energy separately. For routine inference, use the supported calculator or evaluation CLI.

In [ ]:
def raw_forward_for_inspection(calc, atoms, **model_kwargs):
    model = calc.models[0]
    batch = calc._atoms_to_batch(atoms)
    with torch.no_grad():
        output = model(batch.to_dict(), training=False, compute_force=False, **model_kwargs)
    return output

les_calc = MACECalculator(model_paths=str(require_model(les_model_path)), device=inference_device, default_dtype=dtype)
les_output = raw_forward_for_inspection(les_calc, dimer)
for key in ("energy", "les_energy", "latent_charges", "latent_dipoles", "latent_quads", "latent_alphas"):
    value = les_output.get(key)
    if value is not None:
        if torch.is_tensor(value):
            print(key, tuple(value.shape), value.detach().cpu().numpy().reshape(-1)[:8])
        else:
            print(key, value)

Colour predicted charges as latent model variables; do not interpret them as a unique partial-charge analysis.

In [ ]:
latent_q = les_output.get("latent_charges")
if latent_q is None:
    print("This checkpoint did not return latent_charges; inspect its les_arguments and model outputs.")
else:
    q_values = latent_q.detach().cpu().numpy().reshape(len(dimer), -1).sum(axis=1)
    display(show_structure(dimer, values=q_values, value_name="latent charge", value_units="e", title="MACELES latent charges on a water-dimer probe"))
    print("sum of displayed latent charges:", q_values.sum())

### 4.5 Checkpoint demonstration: compare interaction curves

> **Runtime note:** The short inference scan runs on CPU for consistent local and Colab behavior; training still uses CUDA when available.

Plot $E_{AB}-E_A-E_B$ for MACELES, a local-only control and the reference. Match the training set, split, local architecture and fitting protocol; judge both energies and forces across the separation range.

In [ ]:
les_path = require_model(les_model_path)
short_path = require_model(local_model_path)
les_curve_calc = MACECalculator(model_paths=str(les_path), device=inference_device, default_dtype=dtype)
short_curve_calc = MACECalculator(model_paths=str(short_path), device=inference_device, default_dtype=dtype)


def energy_with(calc, atoms):
    atoms = atoms.copy()
    atoms.calc = calc
    return atoms.get_potential_energy()


def monomer_reference_energy(calc):
    # The isolated monomers are identical and do not depend on dimer separation.
    monomer = Atoms("OH2", positions=water_monomer_at((7.0, 8.0, 8.0)),
                    cell=[20.0] * 3, pbc=False)
    return 2.0 * energy_with(calc, monomer)


def interaction_curve(calc, separations):
    monomer_reference = monomer_reference_energy(calc)
    return np.asarray([
        energy_with(calc, water_dimer(float(separation))) - monomer_reference
        for separation in separations
    ])


separations = np.linspace(2.7, 7.0, 18)
interaction_les = interaction_curve(les_curve_calc, separations)
interaction_local = interaction_curve(short_curve_calc, separations)
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(separations, 1000 * interaction_local, "o-", label="matched short-range MACE")
ax.plot(separations, 1000 * interaction_les, "s-", label="MACELES")
ax.axhline(0, color="black", lw=0.8)
ax.set(xlabel="O···O separation (Å)", ylabel="interaction energy (meV)",
       title="Water-dimer interaction: compare matched models")
ax.legend()
ax.grid(alpha=0.25)
plt.show()


#### Reading the interaction scan

Beyond the local cutoff, the local model has no direct cross-fragment message. LES adds electrostatic coupling, but improvement is an empirical question: compare with the reference, check force derivatives and repeat for another orientation.

### 4.6 Training and validation

`les_arguments` selects the latent quantities and solver terms. Energy and force labels can train these internal variables without reference partial-charge labels; that does not make the learned charges unique or directly observable.

The configs below keep the data, local architecture, split and schedule matched. They differ only in whether the LES branch is active. Edit `max_num_epochs` for a shorter demonstration, then run both configs.


In [ ]:
dimer_train = read(asset_root / "data/maceles_water_dimer_train.extxyz", index=":")
dimer_valid = read(asset_root / "data/maceles_water_dimer_valid.extxyz", index=":")
for split_name, frames in (("train", dimer_train), ("validation", dimer_valid)):
    for frame in frames:
        if "REF_energy" not in frame.info or frame.arrays.get("REF_forces", np.empty((0,))).shape != (len(frame), 3):
            raise ValueError(f"{split_name} dimer frame is missing its energy or force labels")
print(f"Dimer labels ready: {len(dimer_train)} train and {len(dimer_valid)} validation frames; energy and forces are present.")

common_config = {
    "loss": "weighted",
    "hidden_irreps": "32x0e + 32x1o + 32x2e",
    "num_interactions": 2,
    "r_max": 5.0,
    "train_file": str(asset_root / "data/maceles_water_dimer_train.extxyz"),
    "valid_file": str(asset_root / "data/maceles_water_dimer_valid.extxyz"),
    "valid_fraction": 0,
    "E0s": {1: 0.0, 8: 0.0},
    "energy_key": "REF_energy",
    "forces_key": "REF_forces",
    "energy_weight": 1.0,
    "forces_weight": 10.0,
    "stress_weight": 0.0,
    "batch_size": 8,
    "valid_batch_size": 8,
    "max_num_epochs": 100,
    "eval_interval": 10,
    "default_dtype": dtype,
    "device": device,
    "seed": 24,
}

les_config = {
    **common_config,
    "name": "MACELES-toy",
    "model": "MACELES",
    "les_arguments": str(asset_root / "config/maceles_toy.yaml"),
    "model_dir": str(asset_root / "models" / "maceles_notebook"),
}
local_config = {
    **common_config,
    "name": "MACELES-short-range-control",
    "model": "MACE",
    "model_dir": str(asset_root / "models" / "maceles_control_notebook"),
}

train_mace(les_config)
train_mace(local_config)

les_model_path = Path(les_config["model_dir"]) / f"{les_config['name']}.model"
local_model_path = Path(local_config["model_dir"]) / f"{local_config['name']}.model"
if not les_model_path.is_file() or not local_model_path.is_file():
    raise FileNotFoundError("One of the matched training runs did not produce its model file")


In [ ]:
read_source("requirements/les.txt", 1, 5, root=source_root)
read_source("mace/modules/extensions.py", 146, 181, root=source_root)

## Continue

Compare this design in [T00 — Extending MACE](T00-Extending-MACE.ipynb). For training, see [Practice I](../MACE_in_practice_I/T01-MACE-Practice-I.ipynb) and [Practice II](../MACE_in_practice_II/T02-MACE-Practice-II.ipynb); for architecture, see [MACE Theory](../MACE_advanced/T03-MACE-Theory.ipynb). Source revisions, example-data limits and acknowledgements are in the [extension guide](README.md).

## Notebook acknowledgements

This extension notebook was made by **Bradley Martin**. The architecture figure is adapted from the MACE Theory figure; see the [figure attribution](figures/ATTRIBUTION.md).